# 新版 LangChain：用 Runnable Memory 做会话记忆

> 这一节演示新版 LangChain 中推荐的 Runnable 记忆方式：`RunnableWithMessageHistory`。

> 结论先行：**记忆不再强依赖旧版 `ConversationBufferMemory`，而是把“历史记录”作为 Runnable 的一部分进行组合**。

## 1. 核心思路（和旧版 Memory 的区别）

新版通常是这 3 个角色：

1. `ChatPromptTemplate + MessagesPlaceholder`：在提示词里预留历史消息位置。
2. `RunnableWithMessageHistory`：把任意 Runnable 包一层，让它自动读写会话历史。
3. `session_id -> history` 映射函数：决定“记忆存哪里、按谁隔离”。

这样你可以很自然地把记忆能力挂在 LCEL 链上，而不是把 memory 对象塞到各种链参数里。

> 生产环境建议把历史记录落到 Redis / 数据库；本节先用内存字典演示原理。

In [5]:
# 2. 环境准备
import os
from typing import Dict

from dotenv import load_dotenv
from pydantic import SecretStr
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

load_dotenv(override=True)

model_name = os.getenv("MODEL")
model_base_url = os.getenv("BASE_URL")
model_api_key = os.getenv("API_KEY")
api_key_value = SecretStr(model_api_key) if model_api_key else None

chat_llm = ChatOpenAI(
    model=model_name or "gpt-4o-mini",
    base_url=model_base_url or None,
    api_key=api_key_value,
    temperature=0
)

print("✅ 模型初始化完成")

✅ 模型初始化完成


In [3]:
# 3. 定义带历史占位符的 Prompt + LCEL 链
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个专业、简洁的中文助理。"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

base_chain = prompt | chat_llm
print("✅ base_chain 创建完成")

✅ base_chain 创建完成


In [ ]:
# 4. 构建 Runnable Memory（关键）
store: Dict[str, ChatMessageHistory] = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """根据 session_id 返回对应的聊天历史对象。"""
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

print("✅ chain_with_memory 创建完成")

✅ chain_with_memory 创建完成


In [8]:
# 5. 演示：同一个 session 会记住历史
session_id = "user_001"

r1 = chain_with_memory.invoke(
    {"input": "你好，我叫小严。"},
    config={"configurable": {"session_id": session_id}},
)
print("第1轮:", r1.content)

r2 = chain_with_memory.invoke(
    {"input": "你还记得我叫什么吗？"},
    config={"configurable": {"session_id": session_id}},
)
print("第2轮:", r2.content)

第1轮: 你好，小严！有什么我可以帮你的吗？
第2轮: 记得，你叫小严。有什么需要我协助的吗？


In [9]:
# 6. 演示：不同 session 之间互不影响（会话隔离）
new_session_id = "user_002"

r3 = chain_with_memory.invoke(
    {"input": "你知道我叫什么吗？"},
    config={"configurable": {"session_id": new_session_id}},
)
print("新会话回答:", r3.content)

print("当前内存中的会话ID:", list(store.keys()))
print("user_001 历史条数:", len(store["user_001"].messages))
print("user_002 历史条数:", len(store["user_002"].messages))

新会话回答: 我不知道您叫什么，我们刚刚开始对话。如果您愿意的话，可以告诉我您的名字，我会记住的。
当前内存中的会话ID: ['user_001', 'user_002']
user_001 历史条数: 4
user_002 历史条数: 2


## 7. 使用 Redis 作为远程记忆存储（服务器版）

你给的服务器地址是 `http://39.106.67.240/`。
需要注意：Redis 连接协议应为 `redis://`（不是 `http://`）。

下面示例默认使用 `39.106.67.240:6379`，并支持从环境变量覆盖：
- `REDIS_URL`（优先）
- `REDIS_PASSWORD`、`REDIS_PORT`、`REDIS_DB`（可选）

In [14]:
# 8. Redis 接口：把会话历史持久化到远程 Redis
from langchain_community.chat_message_histories import RedisChatMessageHistory
load_dotenv(override=True)
# 你给的是 http://39.106.67.240/，Redis 需要 redis:// 协议
redis_host = os.getenv("REDIS_HOST")
redis_port = int(os.getenv("REDIS_PORT", "6379"))
redis_db = int(os.getenv("REDIS_DB", "0"))
redis_password = os.getenv("REDIS_PASSWORD", "")

default_redis_url = (
    f"redis://:{redis_password}@{redis_host}:{redis_port}/{redis_db}"
    if redis_password
    else f"redis://{redis_host}:{redis_port}/{redis_db}"
)
redis_url = os.getenv("REDIS_URL", default_redis_url)
def get_redis_session_history(session_id: str) -> BaseChatMessageHistory:
    return RedisChatMessageHistory(
        session_id=session_id,
        url=redis_url,
        key_prefix="lc:memory:",
    )

chain_with_redis_memory = RunnableWithMessageHistory(
    base_chain,
    get_redis_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

redis_session = "redis_user_001"
try:
    rr1 = chain_with_redis_memory.invoke(
        {"input": "你好，我叫小严。"},
        config={"configurable": {"session_id": redis_session}},
    )
    print("Redis 第1轮:", rr1.content)

    rr2 = chain_with_redis_memory.invoke(
        {"input": "你还记得我叫什么吗？"},
        config={"configurable": {"session_id": redis_session}},
    )
    print("Redis 第2轮:", rr2.content)

    redis_history = get_redis_session_history(redis_session)
    print("Redis 历史条数:", len(redis_history.messages))
except Exception as e:
    print("❌ Redis 连接或调用失败:", repr(e))
    print("请确认 Redis 服务端口、密码和白名单设置是否正确。")

Redis 第1轮: 你好，小严！有什么我可以帮你的吗？
Redis 第2轮: 记得，你叫小严。有什么需要我协助的吗？
Redis 历史条数: 4


In [19]:
rr3 = chain_with_redis_memory.invoke(
    {"input":"清朝持续了多少年？"},
    config={
        "configurable":{
            "session_id": redis_session
        },
        "tags": ["test", "redis"]
    }

)
print("Redis 第3轮:", rr3.content)

Redis 第3轮: 清朝从1644年建立到1912年结束，共持续了268年。


In [21]:
print("Redis 历史条数:", len(redis_history.messages))
print(redis_history.messages[-1])

Redis 历史条数: 8
content='清朝从1644年建立到1912年结束，共持续了268年。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 163, 'total_tokens': 187, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen3-coder-plus', 'system_fingerprint': None, 'id': 'chatcmpl-e7aacdd7-0287-4271-9640-257180502a5a', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d292b-729c-77d2-8c6b-4e1324813084-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 163, 'output_tokens': 24, 'total_tokens': 187, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


## 9. 你可以怎么扩展到生产环境

- 把 `ChatMessageHistory` 换成 `RedisChatMessageHistory` 或数据库实现，保持 `get_session_history` 接口不变。
- 在 `session_id` 之外增加租户ID、用户ID，实现更细粒度隔离。
- 对历史消息做裁剪（窗口化）或摘要，降低 token 消耗。
- 如果你在做复杂 Agent 工作流，也可以考虑 LangGraph 的持久化记忆方案。

---

到这里，你已经掌握了新版 LangChain 在 LCEL 里做记忆的主流方式：
**`prompt | model` + `RunnableWithMessageHistory` + `session_id`**。

## 10. 流式输出（Streaming）

如果你希望模型一边生成一边打印，可以用 `stream()`。
下面示例使用 `chain_with_memory`，同样支持会话记忆。

In [ ]:
# 11. Runnable + Memory 的流式输出示例
from langchain_core.messages import AIMessageChunk

stream_session = "stream_user_001"
stream_question = "请用 5 条要点介绍 LangChain 的 Runnable，并给一个简短示例。"


# chunk: AIMessageChunk
print("助手（流式）: ", end="", flush=True)
for chunk in chain_with_redis_memory.stream(
    {"input": stream_question},
    config={"configurable": {"session_id": stream_session}},
):
    chunk: AIMessageChunk
    piece = getattr(chunk, "content", "")
    if isinstance(piece, str) and piece:
        print(piece, end="", flush=True)
    elif isinstance(piece, list):
        for part in piece:
            if isinstance(part, dict):
                text = part.get("text", "")
                if text:
                    print(text, end="", flush=True)
print("\n\n✅ 流式输出完成")

# 再问一轮，验证流式 + 记忆可以同时工作
print("助手（第2轮流式）: ", end="", flush=True)
for chunk in chain_with_redis_memory.stream(
    {"input": "请复述我上一轮问题的主题。"},
    config={"configurable": {"session_id": stream_session}},
):
    chunk: AIMessageChunk
    piece = getattr(chunk, "content", "")
    if isinstance(piece, str) and piece:
        print(piece, end="", flush=True)
    elif isinstance(piece, list):
        for part in piece:
            if isinstance(part, dict):
                text = part.get("text", "")
                if text:
                    print(text, end="", flush=True)
print("\n\n✅ 第2轮流式输出完成")

助手（流式）: LangChain 的 Runnable 是一个核心概念，用于构建可执行的链式操作。以下是其五个关键要点：

1. **统一接口**：Runnable 提供了一个标准接口，允许不同组件（如 LLM、工具、提示等）以一致的方式组合和执行。

2. **链式调用**：支持将多个组件串联成一个流程，前一个输出自动作为后一个输入，简化了复杂逻辑的实现。

3. **异步支持**：内置异步执行能力，提升 I/O 密集型任务（如 API 调用）的性能。

4. **灵活组合**：可通过 `|` 操作符或 `chain()` 方法轻松组合多个 Runnable 对象，支持条件分支与并行处理。

5. **易于扩展**：开发者可自定义 Runnable 子类，封装特定业务逻辑并无缝集成到现有链中。

### 简短示例

```python
from langchain_core.runnables import RunnableLambda
from langchain_openai import OpenAI

# 定义一个简单处理函数
def add_prefix(x):
    return f"Processed: {x}"

# 创建 Runnable 链
chain = RunnableLambda(add_prefix) | OpenAI()

# 执行
result = chain.invoke("Hello")
```

此示例先对输入添加前缀，再传递给语言模型生成响应。

✅ 流式输出完成
助手（第2轮流式）: 您上一轮问题的主题是：**请用 5 条要点介绍 LangChain 的 Runnable，并给一个简短示例**。

✅ 第2轮流式输出完成
